In [ ]:
# ================================
# Cell 1 — Imports & helpers
# ================================
import os, sys
project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
import json
from typing import Dict, Any, List, Tuple

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Your prompt helpers:
# - counterfactual_CoT_prompt.generate_messages(cot: str, increase: bool=True) -> List[dict]
# - paraphrase_CoT_prompt.generate_messages(cot: str) -> List[dict]
from prompts import main_prompt, counterfactual_CoT_prompt, paraphrase_CoT_prompt


def safe_split_cot(model_response: str) -> Tuple[str, str]:
    """
    Splits a full model response into (cot_without_final_answer, tail_after_marker).
    If the marker is missing, treats the whole response as CoT and empty tail.
    """
    parts = model_response.split("FINAL ANSWER:", maxsplit=1)
    if len(parts) == 1:
        cot = parts[0].strip()
        tail = ""
    else:
        cot = parts[0].rstrip()
        tail = parts[1].strip()
    return cot, tail


def choose_opposite_direction(answer: int) -> bool:
    """
    Heuristic for 'opposite' direction:
      - If answer < 5, push 'increase=True' (nudge upward)
      - If answer >= 5, push 'increase=False' (nudge downward)
    Returns the 'increase' flag expected by counterfactual_CoT_prompt.generate_messages.
    """
    try:
        a = int(answer)
    except Exception:
        return False
    return True if a < 5 else False


def chunked(iterable, n):
    """Yield successive n-sized chunks from a list."""
    for i in range(0, len(iterable), n):
        yield iterable[i:i + n]


def call_qwen_batch(
    model,
    tokenizer,
    list_of_messages: List[List[Dict[str, str]]],
    max_new_tokens: int = 1024
) -> List[str]:
    """
    Runs Qwen generation for a batch of chat 'messages' lists.
    Returns a list of decoded generations (one per input).
    """
    prompts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        for msgs in list_of_messages
    ]

    model_inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        padding_side='left'
    )

    device = next(model.parameters()).device
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            do_sample=False,  # deterministic
            temperature=None,
            top_p=None,
            top_k=None,
            max_new_tokens=max_new_tokens
        )

    # Slice off each prompt's input tokens
    out_ids = []
    for i in range(generated_ids.size(0)):
        input_len = model_inputs["input_ids"][i].size(0)
        out_ids.append(generated_ids[i, input_len:])

    return tokenizer.batch_decode(out_ids, skip_special_tokens=True)


# Optional: wire your model here if you want to compute downstream answers.
def get_model_answer_with_injected_cot(item: Dict[str, Any], injected_cot: str) -> int:
    """
    TODO: Replace this stub with your model’s inference call that conditions on injected_cot.
    Return the new predicted answer (0-9).
    """
    raise NotImplementedError


In [ ]:
# ================================
# Cell 2 — Core processing
# ================================
from tqdm import tqdm
import math
def process_data(
    data: Dict[str, Dict[str, Any]],
    batch_size: int = 8,
    model_id: str = "Qwen/Qwen3-4B-Instruct-2507",
    compute_new_answers: bool = False,
    force_direction: str = "auto",  # "auto" | "increase" | "decrease"
    max_new_tokens: int = 1024
) -> Dict[str, Dict[str, Any]]:
    """
    - Takes a dict of items (each value has 'model_response', 'answer', etc.)
    - Adds 'counterfactual_response' and 'paraphrased_response' to each item
    - Optionally computes 'counterfactual_answer' and 'paraphrased_answer' (if you implement the stub)
    - Returns the updated dict
    """
    keys = list(data.keys())

    # Load Qwen
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.eval()

    # Build message queues
    to_counterfactual = []  # (key, messages)
    to_paraphrase = []      # (key, messages)

    for k in keys:
        item = data[k]
        model_response = item["model_response"].split('assistant')[-1]
        cot, _ = safe_split_cot(model_response)

        if force_direction == "increase":
            inc = True
        elif force_direction == "decrease":
            inc = False
        else:
            inc = choose_opposite_direction(item.get("answer", 0))

        cf_msgs = counterfactual_CoT_prompt.generate_messages(cot, increase=inc)
        pp_msgs = paraphrase_CoT_prompt.generate_messages(cot)

        to_counterfactual.append((k, cf_msgs))
        to_paraphrase.append((k, pp_msgs))

    # Run counterfactuals in batches
    for batch in tqdm(chunked(to_counterfactual, batch_size), desc='Counterfactuals..', total=math.ceil(len(to_counterfactual) / batch_size)):
        batch_keys, batch_msgs = zip(*batch)
        generations = call_qwen_batch(model, tokenizer, list(batch_msgs), max_new_tokens=max_new_tokens)
        for k, gen in zip(batch_keys, generations):
            data[k]["counterfactual_response"] = gen.strip() + '\nFINAL ANSWER:'

    # Run paraphrases in batches
    for batch in tqdm(chunked(to_paraphrase, batch_size), desc='Paraphrased...', total=math.ceil(len(to_paraphrase) / batch_size)):
        batch_keys, batch_msgs = zip(*batch)
        generations = call_qwen_batch(model, tokenizer, list(batch_msgs), max_new_tokens=max_new_tokens)
        for k, gen in zip(batch_keys, generations):
            data[k]["paraphrased_response"] = gen.strip() + '\nFINAL ANSWER:'

    # Optionally: feed back into your CoT-conditioned model
    if compute_new_answers:
        for k in keys:
            item = data[k]
            try:
                if "counterfactual_response" in item:
                    item["counterfactual_answer"] = int(
                        get_model_answer_with_injected_cot(item, item["counterfactual_response"])
                    )
                if "paraphrased_response" in item:
                    item["paraphrased_answer"] = int(
                        get_model_answer_with_injected_cot(item, item["paraphrased_response"])
                    )
            except NotImplementedError:
                pass
            except Exception as e:
                item["cot_injection_error"] = str(e)

    return data


In [ ]:
import json
INPUT_JSON = "/work/wildfirerisk/grpo-checkpoint-1800-predictions.json"
with open(INPUT_JSON, 'r') as f:
    x = json.load(f)
for k, v in x.items():
    print(v['model_response'].split('assistant')[-1])
    break

In [ ]:
# ================================
# Cell 3 — Example: load, run, save
# ================================
# Set your paths (or skip file I/O and pass a dict directly to process_data)

INPUT_JSON = "/work/wildfirerisk/vlm_cot_predictions.json"   # path to your existing dict JSON
OUTPUT_JSON = "/work/wildfirerisk/vlm_cot_predictions_complete.json"

# Load dict
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)
new_data = {}
for k, v in data.items():
    if 'test' in v['img_path']:
        new_data[k] = v
print("Total samples", len(new_data))
# Process
updated = process_data(
    new_data,
    batch_size=128,
    model_id="Qwen/Qwen3-4B-Instruct-2507",
    compute_new_answers=False,
    force_direction="auto",      # or "increase" / "decrease"
    max_new_tokens=1024
)

# Save back
with open(OUTPUT_JSON, "w", encoding="utf-8") as wf:
    json.dump(updated, wf, ensure_ascii=False, indent=2)

print(f"Done. Wrote: {OUTPUT_JSON}")


In [ ]:
# --- Single-cell completion of paraphrased & counterfactual CoTs ---

import re, json
from pathlib import Path
from typing import Dict, Any, List, Optional
from PIL import Image as PILImage
from tqdm import tqdm
from config import DATA_DIR
import torch
from transformers import AutoModelForImageTextToText, Qwen2_5_VLProcessor

# ----- Fallbacks if not defined earlier -----
VLM_CKPT = "/work/wildfirerisk/trainings/grpo_1/checkpoint-1800"
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
BATCH_SIZE = 40
MAX_NEW_TOKENS = 1024
DEVICE_STR = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_PATH = f'{DATA_DIR}/vlm_cot_predictions_complete.json'
CLIMATE_JSON = f'{DATA_DIR}/climate_data.json' 
with open(CLIMATE_JSON, 'r') as f:
    climate = json.load(f)
# ----- Minimal helpers -----
FINAL_RE = re.compile(r"FINAL ANSWER:\s*([0-9])\s*$", re.IGNORECASE)
def parse_final_digit(text: str, default: int = -1) -> int:
    text = text.strip()
    m = FINAL_RE.search(text)
    if m:
        return int(m.group(1))
    digits = re.findall(r"([0-9])", text)
    return int(digits[-1]) if digits else int(default)

_COORD_RE = re.compile(r"lon(?P<lon>-?\d+(?:\.\d+)?)_lat(?P<lat>-?\d+(?:\.\d+)?)")
def parse_key_from_path(p: str) -> Optional[str]:
    m = _COORD_RE.search(Path(p).name)
    if not m: return None
    lon, lat = float(m.group("lon")), float(m.group("lat"))
    return f"{lat}_{lon}"

def build_user_text(climate_key: str) -> str:
    return main_prompt.build_prompt(climate[climate_key])

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# ----- Load processor & model -----
processor = Qwen2_5_VLProcessor.from_pretrained(BASE_MODEL_ID, use_fast=True, padding_side="left")
vlm = AutoModelForImageTextToText.from_pretrained(
    VLM_CKPT,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).to(DEVICE_STR).eval()

# ----- Collect work items (paraphrase + counterfactual) -----
def collect_work(field_name: str) -> List[Dict[str, Any]]:
    work = []
    for k, item in updated.items():
        injected = item.get(field_name)
        img_path = item.get("img_path")
        if not injected or not img_path:
            continue
        climate_key = item.get("key") or parse_key_from_path(img_path)
        if not climate_key or climate_key not in climate:
            continue
        work.append({"k": k, "injected": injected, "img_path": img_path, "climate_key": climate_key})
    return work

def run_completion_for_field(field_name: str, out_completed: str, out_answer: str):
    work = collect_work(field_name)
    if not work:
        print(f"[{field_name}] nothing to do.")
        return

    print(f"[{field_name}] {len(work)} items | batch_size={BATCH_SIZE}")
    with torch.inference_mode():
        for batch in tqdm(list(chunks(work, BATCH_SIZE)), total=(len(work)+BATCH_SIZE-1)//BATCH_SIZE, desc=f"{field_name} → complete"):
            # Build prompts and images
            prompts, images, keys = [], [], []
            for w in batch:
                user_text = build_user_text(w["climate_key"])
                conversation = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image"},
                            {"type": "text", "text": user_text},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": w["injected"].rstrip() + "\nFINAL ANSWER:"}
                        ],
                    },
                ]
                prompt = processor.apply_chat_template(
                    conversation,
                    tokenize=False,
                    continue_final_message=True,  # <— continue the assistant turn
                )
                prompts.append(prompt)
                images.append(PILImage.open(w["img_path"]).convert("RGB"))
                keys.append(w["k"])

            inputs = processor(images=images, text=prompts, padding=True, return_tensors="pt")
            inputs = {k:(v.to(DEVICE_STR) if hasattr(v, "to") else v) for k,v in inputs.items()}

            gen_ids = vlm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            texts = processor.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

            for k_id, txt in zip(keys, texts):
                updated[k_id][out_completed] = txt.strip()
                updated[k_id][out_answer] = int(parse_final_digit(txt, default=-1))

            del inputs, gen_ids, images, prompts

# ----- Run both passes & save -----
run_completion_for_field("paraphrased_response", "paraphrased_completed", "paraphrased_answer")
run_completion_for_field("counterfactual_response", "counterfactual_completed", "counterfactual_answer")

with open(SAVE_PATH, 'w') as f:
    json.dump(updated, f)
print(f"Done. Saved to {SAVE_PATH}")
